# Unified Service Scheduler — Live Scheduling Demo

Explore a working automotive appointment scheduler. See how the application finds available slots, assigns qualified technicians and compatible service bays, prevents double-booking and handles simultaneous requests.

### Get started

1. Select a **CPU runtime** in Google Colab or Kaggle; enable Internet access.
2. Click **Run All**.
3. Wait until **Cell 23 — Live Web Demo** displays **Open Live Scheduling Demo ↗**. Open it in a new browser tab. The final cell keeps running while the demo is available.

**Try it yourself:** Browse service availability, inspect appointment records, and run **Test Booking Conflict (2 requests)** or **Simulate 10 Concurrent Requests**. These activities need **no token**. Creating or cancelling an appointment in the live interface requires a private **demo write token** from the owner (see Cells 20–21).

### What this demonstrates

| Capability | What to look for |
| --- | --- |
| Availability | Technician skills, bay compatibility, service duration and dealership hours determine bookable slots |
| Confirmation | Each accepted appointment has assigned resources |
| Conflict prevention | Two requests cannot reserve the same limited capacity |
| Concurrency | One competing request wins; the rest are rejected |
| Persistence | Bookings and cancellation records remain after refresh |

This notebook uses **synthetic data and an isolated SQLite database**. It does not modify a production database or require a GPU, Docker, Google Drive or GitHub credentials.

### How the session works

The notebook first runs its automated checks. **Cell 23** then opens a temporary public Cloudflare link for **15 minutes by default** and shows sanitized request activity until the session ends. The URL changes when you restart the live session. Viewing and simulations are available without a token; booking and cancellation require authorization.

**Navigation:** 23 numbered cells, including **10 Python code cells**. The owner-only token control is **Cell 21**; the Live Demo is **Cell 23**. If you already started the demo and need the token, follow the instructions in **Cell 20**.

**Requirements:** Python 3, Git, Internet access and Node.js 24+. The setup installs Node.js 24 automatically on supported Colab runtimes; other environments must provide a compatible version.


## Cell 02 — Prepare the demo environment

The next cell retrieves the public project source, installs its dependencies and prepares a **fresh isolated database** populated with sample dealerships, customers, vehicles, technicians and service bays.

**What you should see:** successful installation and setup, plus the detected platform, application revision and workspace path for reproducibility. No manual configuration is required for standard Colab and Kaggle runtimes.

Existing application databases are not overwritten.


In [ ]:
# Cell 03 of 23 — Prepare isolated demo data
from pathlib import Path
import subprocess, urllib.request, urllib.error, json, os, time, uuid, datetime
import shutil, socket, signal, atexit, sys, re

REPOSITORY_URL = "https://github.com/dangkhoa2016/Keyloop-Scenario-A-Unified-Service-Scheduler.git"
if os.environ.get("KEYLOOP_LAB_WORKSPACE"):
    WORKSPACE = Path(os.environ["KEYLOOP_LAB_WORKSPACE"]).expanduser().resolve()
    PLATFORM = "Custom Jupyter workspace"
elif Path("/kaggle/working").is_dir():
    WORKSPACE = Path("/kaggle/working")
    PLATFORM = "Kaggle"
elif Path("/content").is_dir():
    WORKSPACE = Path("/content")
    PLATFORM = "Google Colab"
else:
    WORKSPACE = Path.cwd().resolve()
    PLATFORM = "Local Jupyter"

WORKSPACE.mkdir(parents=True, exist_ok=True)
assert shutil.which("git"), "Git is required to clone the public repository"

def get_node_major():
    if not shutil.which("node"):
        return 0
    result = subprocess.run(["node", "--version"], capture_output=True, text=True, timeout=15)
    if result.returncode:
        return 0
    match = re.fullmatch(r"v(\d+)\.\d+\.\d+", result.stdout.strip())
    return int(match.group(1)) if match else 0

if get_node_major() < 24:
    if PLATFORM == "Google Colab" and hasattr(os, "geteuid") and os.geteuid() == 0:
        # Google Colab images are updated independently of Node.js; install a supported LTS.
        setup = WORKSPACE / ("nodesource-node24-setup-" + uuid.uuid4().hex[:8] + ".sh")
        try:
            with urllib.request.urlopen("https://deb.nodesource.com/setup_24.x", timeout=30) as response:
                setup.write_bytes(response.read())
            subprocess.run(["bash", str(setup)], check=True, timeout=360)
            subprocess.run(["apt-get", "install", "-y", "nodejs"], check=True, timeout=360)
        finally:
            setup.unlink(missing_ok=True)
    else:
        raise RuntimeError("Node.js 24+ is required. Install it in this runtime and Run All again.")
assert get_node_major() >= 24, "Node.js 24+ installation did not succeed"
assert shutil.which("npm"), "npm is required"
print("PLATFORM=", PLATFORM, "NODE_VERSION=", subprocess.check_output(["node", "--version"], text=True).strip())
print("WORKSPACE=", WORKSPACE)

# Prefer a fresh public checkout to avoid stale code from earlier notebook runs.
provided = os.environ.get("KEYLOOP_NOTEBOOK_SOURCE")
if provided:
    SOURCE_ROOT = Path(provided).expanduser().resolve()
    assert (SOURCE_ROOT / "package-lock.json").is_file(), "Invalid KEYLOOP_NOTEBOOK_SOURCE"
    print("LOCAL_SOURCE_REUSED=PASS", SOURCE_ROOT)
else:
    SOURCE_ROOT = WORKSPACE / ("keyloop-a-public-source-" + uuid.uuid4().hex[:10])
    clone_env = {k:v for k,v in os.environ.items()
                 if k not in ("GH_TOKEN", "GITHUB_TOKEN", "GIT_ASKPASS", "GIT_ASKPASS_REQUIRE")}
    clone_env["GIT_TERMINAL_PROMPT"] = "0"
    clone = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY_URL, str(SOURCE_ROOT)],
        text=True, capture_output=True, timeout=150, env=clone_env
    )
    assert clone.returncode == 0, "Public GitHub clone failed: " + clone.stderr[-500:]
    print("PUBLIC_SOURCE_CLONE=PASS", SOURCE_ROOT)
rev = subprocess.run(["git", "rev-parse", "HEAD"],
                     cwd=SOURCE_ROOT, capture_output=True, text=True, timeout=15)
assert rev.returncode == 0, "Cannot determine the cloned source revision"
SOURCE_REVISION = rev.stdout.strip()
print("SOURCE_REVISION=", SOURCE_REVISION)
print("DEMO_INTERFACE=Unified Service Scheduler reviewer demo")

def matches_restored_ui(body):
    """Recognize the validated, original scheduling UI without its retired header."""
    required = (
        "New to this demo?",
        "Booking access &amp; help",
        "Reviewer Test Guide →",
        "Read-only preview · Booking access required",
    )
    forbidden = (
        "KEYLOOP / SCENARIO A",
        "OPEN-SOURCE SCHEDULING LAB",
        'data-ui-variant="community-lab"',
        "Explore the scheduler",
    )
    return all(fragment in body for fragment in required) and all(
        fragment not in body for fragment in forbidden
    )

def verify_live_source_is_current():
    """Refuse to publish an outdated UI from a prior Colab notebook session."""
    ui_file = ROOT / "public" / "js" / "main.js"
    expected_marker = 'New to this demo?'
    if not ui_file.is_file() or not matches_restored_ui(ui_file.read_text(encoding="utf-8")):
        print("LIVE_UI_PREFLIGHT=FAIL outdated_frontend", flush=True)
        raise RuntimeError(
            "The demo interface could not be validated. "
            "Stop the previous tunnel, restart the Colab runtime, then Run all "
            "from the first cell. Do not rerun only the Live Logs cell."
        )
    # Check the remote branch immediately before exposing the public tunnel.
    remote = subprocess.run(
        ["git", "ls-remote", REPOSITORY_URL, "refs/heads/main"],
        capture_output=True, text=True, timeout=35,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    )
    if remote.returncode != 0 or not remote.stdout.strip():
        raise RuntimeError(
            "Could not verify GitHub main before publishing the tunnel. "
            "Check network access and Run all again."
        )
    remote_revision = remote.stdout.strip().split()[0]
    if SOURCE_REVISION != remote_revision:
        print("LIVE_UI_PREFLIGHT=FAIL stale_source", flush=True)
        print("CLONED_SOURCE_REVISION=", SOURCE_REVISION, flush=True)
        print("CURRENT_GITHUB_MAIN=", remote_revision, flush=True)
        raise RuntimeError(
            "GitHub main has changed since this notebook session cloned it. "
            "Stop the old live tunnel, restart runtime, and Run all from the beginning "
            "to obtain the latest HTML and JavaScript assets."
        )
    print("LIVE_UI_PREFLIGHT=PASS", "source=" + SOURCE_REVISION[:12],
          "frontend=reviewer-demo", flush=True)
    return remote_revision

# Ensure browsers cannot reuse an older service-worker/browser-cached main.js.
# This applies only to the disposable sandbox, never to the Git checkout.
def prepare_versioned_frontend():
    ui = ROOT / "public" / "js" / "main.js"
    index = ROOT / "public" / "index.html"
    assert ui.is_file() and index.is_file(), "Expected Vue web UI files are absent"
    import hashlib
    asset_ui = SOURCE_ROOT / "public" / "js" / "Reviewer-UI.js"
    asset_bytes = asset_ui.read_bytes()
    assert hashlib.sha256(asset_bytes).hexdigest() == "99a70c7dc230dc1d04c3b6dc285e59ff2154ebd139461fec7f6b98ea0d809424", "Reviewer UI asset checksum mismatch"
    ui.write_bytes(asset_bytes)
    body = ui.read_text(encoding="utf-8")
    # Kaggle .593.log golden-reference footer class, applied ONLY to the Colab
    # throwaway working copy; never modify source checkout or GitHub main.
    old_footer = 'class="text-slate-400 text-sm">Demonstration with synthetic data.'
    new_footer = 'class="text-slate-500 text-sm">Demonstration with synthetic data.'
    assert body.count(old_footer)==1, "The frontend styling does not match the supported demo source"
    body = body.replace(old_footer,new_footer)
    ui.write_text(body,encoding="utf-8")
    if not matches_restored_ui(body):
        raise RuntimeError("The demo interface is not compatible with this notebook; public launch was cancelled")
    html_body = index.read_text(encoding="utf-8")
    old = 'src="/js/main.js"'
    assert old in html_body, "Cannot locate frontend module script in index.html"
    index.write_text(html_body.replace(old, 'src="/js/main.js?v=' + SOURCE_REVISION[:16] + '"'), encoding="utf-8")
    print("FRONTEND_VERSIONED_ASSET=PASS", SOURCE_REVISION[:16], flush=True)

# Fresh SQLite/database and server, never the public demo or user's existing source.
ROOT = WORKSPACE / ("keyloop-a-lab-isolated-" + uuid.uuid4().hex[:10])
assert ROOT != SOURCE_ROOT and not ROOT.is_relative_to(SOURCE_ROOT)
assert not ROOT.exists()
exclude = shutil.ignore_patterns(
    ".git", ".runtime", ".env", ".env.*", "node_modules",
    "data", "backups", "evidence", "__pycache__", "*.sqlite*",
    "*.log", "*.tar.gz", "*.zip", "*token*", "*secret*"
)
shutil.copytree(SOURCE_ROOT, ROOT, ignore=exclude)
prepare_versioned_frontend()
(ROOT / "data").mkdir()
(ROOT / "evidence").mkdir()

with socket.socket() as probe:
    probe.bind(("127.0.0.1", 0))
    LAB_PORT = probe.getsockname()[1]
assert LAB_PORT != 3000
sandbox_env = os.environ.copy()
for forbidden in ("DEMO_WRITE_TOKEN", "PUBLIC_DEMO", "DATABASE_PATH",
                  "HOST", "PORT", "GH_TOKEN", "GITHUB_TOKEN", "KAGGLE_KEY", "KAGGLE_API_TOKEN"):
    sandbox_env.pop(forbidden, None)
sandbox_env.update(
    HOST="127.0.0.1",
    PORT=str(LAB_PORT),
    DATABASE_PATH=str(ROOT / "data" / "scheduler.sqlite")
)
assert Path(sandbox_env["DATABASE_PATH"]).resolve().is_relative_to(ROOT)
# Match the original protected Kaggle reviewer interface in the Colab public demo.
# The credential is local to this disposable notebook run and is NEVER published.
import secrets
LAB_TOKEN_DIR = ROOT / ".runtime"
LAB_TOKEN_DIR.mkdir(mode=0o700,exist_ok=True)
LAB_TOKEN_FILE = LAB_TOKEN_DIR / "demo-write-token"
LAB_TOKEN = secrets.token_urlsafe(32)
LAB_TOKEN_FILE.write_text(LAB_TOKEN,encoding="utf-8")
LAB_TOKEN_FILE.chmod(0o600)
# Owner-only pointer: stores the FILE PATH, never the token itself.
if PLATFORM == "Google Colab":
    token_pointer = WORKSPACE / ".keyloop-current-token-path"
    token_pointer.write_text(str(LAB_TOKEN_FILE),encoding="utf-8")
    token_pointer.chmod(0o600)
sandbox_env["DEMO_WRITE_TOKEN"] = LAB_TOKEN
sandbox_env["PUBLIC_DEMO"] = "1"
# Tests must NOT inherit the public demo token guard.
test_env = sandbox_env.copy()
for key in ("DEMO_WRITE_TOKEN", "PUBLIC_DEMO"):
    test_env.pop(key, None)
assert "DEMO_WRITE_TOKEN" not in test_env and "PUBLIC_DEMO" not in test_env
print("REVIEWER_ACCESS=PROTECTED; private token stored securely")

server = None
def stop_lab_server():
    global server
    if server is not None and server.poll() is None:
        try:
            os.killpg(server.pid, signal.SIGTERM)
            server.wait(timeout=5)
        except Exception:
            try: server.kill()
            except Exception: pass
atexit.register(stop_lab_server)

def run(*argv):
    p = subprocess.run(argv, cwd=ROOT, env=sandbox_env,
                       text=True, capture_output=True, timeout=300)
    print("$", *argv, "status:", p.returncode, "\n", p.stdout[-2500:], p.stderr[-500:])
    assert p.returncode == 0, f"Command failed: {argv}"

print("LAB_SANDBOX=", ROOT)
print("LAB_LOOPBACK_PORT=", LAB_PORT)
print("ISOLATED_DATABASE=", sandbox_env["DATABASE_PATH"])
run("npm", "ci")
run("npm", "run", "db:migrate")
run("npm", "run", "db:seed:rich")
run("npm", "run", "db:seed:demo")

## Cell 04 — Verify the scheduling rules

Automated tests check that bookings use qualified technicians and compatible service bays throughout each appointment, respect resource availability and prevent overlapping assignments.

They also cover duplicate submissions, conflict responses, persistence and concurrent booking requests.

**Expected result:** all scheduling tests pass. The interactive browser demo comes later in Cell 23.


In [ ]:
# Cell 05 of 23 — Run scheduling tests
p=subprocess.run(["npm","test"],cwd=ROOT,env=test_env,text=True,capture_output=True)
print(p.stdout[-7000:])
assert p.returncode==0, p.stderr[-1000:]

## Cell 06 — Start the scheduling service

The next cell starts the application on a local address using your temporary SQLite database and waits until the API is ready.

**Expected result:** the local service responds successfully. You do not need to open its private address; a public browser link appears in Cell 23.


In [ ]:
# Cell 07 of 23 — Start local scheduling API
BASE=f"http://127.0.0.1:{LAB_PORT}"
def api(path,method="GET",payload=None,headers=None):
    req=urllib.request.Request(
        BASE+path,method=method,
        data=None if payload is None else json.dumps(payload).encode(),
        headers={"Content-Type":"application/json",**({"X-Demo-Write-Token":LAB_TOKEN} if method != "GET" else {}),**(headers or {})}
    )
    try:
        with urllib.request.urlopen(req,timeout=15) as r: return r.status,json.load(r)
    except urllib.error.HTTPError as e: return e.code,json.load(e)
log=open(ROOT/"evidence"/"notebook-server.log","a")
server=subprocess.Popen(
    ["node","src/server.js"],cwd=ROOT,env=sandbox_env,
    stdout=log,stderr=subprocess.STDOUT,start_new_session=True
)
log.close()
for _ in range(50):
    if server.poll() is not None:
        raise RuntimeError("Isolated server terminated at startup")
    try:
        if api("/readyz")[0]==200: break
    except Exception:
        pass
    time.sleep(0.2)
else:
    raise RuntimeError("Isolated backend readiness timeout")
assert api("/readyz")[0]==200
print("ISOLATED_BACKEND_READY=",BASE)
# HTTP-level assertion: the running Hono server must serve the updated Vue UI,
# not merely contain it on disk. Also verify the versioned script path.
with urllib.request.urlopen(BASE + "/", timeout=15) as r:
    actual_html = r.read().decode("utf-8")
assert "/js/main.js?v=" + SOURCE_REVISION[:16] in actual_html, "Stale index.html response"
with urllib.request.urlopen(BASE + "/js/main.js?v=" + SOURCE_REVISION[:16], timeout=15) as r:
    actual_js = r.read().decode("utf-8")
assert matches_restored_ui(actual_js), "HTTP_FRONTEND_MISMATCH: expected restored original interface"
print("ORIGIN_HTTP_FRONTEND=PASS", SOURCE_REVISION[:16], flush=True)


## Cell 08 — Book an appointment and prevent duplicates

This example demonstrates a complete booking flow through the application API:

1. Find an available time.
2. Confirm an appointment (**HTTP 201**).
3. Repeat the same request using its idempotency key: the existing booking is returned, **not duplicated** (**HTTP 200**).
4. Send a new competing request for the occupied slot: it is rejected (**HTTP 409**).
5. Read the saved appointment to confirm persistence (**HTTP 200**).

All changes remain inside this notebook's temporary database.


In [ ]:
# Cell 09 of 23 — Exercise booking and conflict
from urllib.parse import urlencode
today=datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=7))).date()
date=(today+datetime.timedelta(days=(7-today.weekday())%7 or 7)).isoformat()
query=urlencode({"dealershipId":"dlr_001","serviceTypeId":"svc_brake","date":date})
s,found=api("/api/v1/availability?"+query)
assert s==200 and found["data"]["slots"]
body={"customerId":"cus_001","vehicleId":"veh_001","dealershipId":"dlr_001","serviceTypeId":"svc_brake","startAt":found["data"]["slots"][0]["startAt"]}
key="lab-"+uuid.uuid4().hex
one,created=api("/api/v1/appointments","POST",body,{"Idempotency-Key":key})
two,replayed=api("/api/v1/appointments","POST",body,{"Idempotency-Key":key})
three,conflict=api("/api/v1/appointments","POST",body,{"Idempotency-Key":"lab-"+uuid.uuid4().hex})
four,persisted=api("/api/v1/appointments/"+created["data"]["id"])
report={"status":[one,two,three,four],"appointmentId":created["data"]["id"],"sameId":created["data"]["id"]==replayed["data"]["id"]}
print(json.dumps(report,indent=2))
assert report["status"]==[201,200,409,200] and report["sameId"]

## Cell 10 — Check the web application

The next cell verifies that the local server serves both the homepage and the JavaScript needed by the browser.

**Expected result:** HTTP `200` for both requests. The address printed here is local to the runtime. To interact from your own browser, use the public URL created in Cell 23.

For the best experience, open the demo with Internet access in an up-to-date desktop or mobile browser.


In [ ]:
# Cell 11 of 23 — Verify homepage and assets
run("npm","run","evidence")
print("GET /:",urllib.request.urlopen(BASE+"/").status)
print("GET /js/main.js:",urllib.request.urlopen(BASE+"/js/main.js").status)
print("Sandbox-only URL (not public):",BASE)

## Cell 12 — Suggested reviewer walkthrough

Use these steps once the live link is available in Cell 23:

1. **Check availability:** choose a dealership, customer, vehicle, service and date, then explore the offered slots.
2. **Inspect appointments:** filter by status or dealership; expand an appointment to see the assigned technician and service bay.
3. **Test a booking conflict:** click **Test Booking Conflict (2 requests)**. One request should succeed and one should be rejected.
4. **Test concurrency:** click **Simulate 10 Concurrent Requests**. Only one should be confirmed for the constrained resources. Allow for the simulator cooldown.
5. **Optional owner-authorized booking:** enter and verify the private write token, confirm an appointment, refresh the page to check persistence, then cancel it to observe the status change.

The conflict and concurrency simulations use an independent disposable database. They do not add appointments to the main demo list and do not require a token.


## Cell 13 — See the scheduler handle simultaneous requests

The next cell runs both conflict scenarios against the API: **two requests** and **ten concurrent requests** competing for restricted technician and bay capacity.

**Expected result:** each scenario confirms **one request** and rejects the others. The simulation also checks its own database integrity. The second scenario waits for the API cooldown before starting.

These experiments use isolated test data and do not change the interactive demo's appointment list.


In [ ]:
# Cell 14 of 23 — Run concurrency simulations
p=subprocess.run(["npm","run","evidence"],cwd=ROOT,env=sandbox_env,text=True,capture_output=True)
print(p.stdout[-4000:])
assert p.returncode==0, p.stderr[-1000:]
for idx,mode in enumerate(("conflict","concurrent")):
    if idx: time.sleep(21)  # backend 20-second simulation cooldown
    status,result=api("/api/v1/demo/simulate",method="POST",payload={"mode":mode})
    assert status==200, (status,result)
    data=result.get("data",result)
    print(mode,data.get("confirmed"),data.get("rejected"),data.get("databaseConfirmed"),data.get("integrity"))
    assert data["confirmed"]==1 and data["rejected"]==(1 if mode=="conflict" else 9)
    assert data["databaseConfirmed"]==1 and data["integrity"]=="ok"

## Cell 15 — Accessibility and device coverage

The application is designed for desktop and mobile browsers. During the live demo, you can explore keyboard navigation, modal dialogs and layout at different screen widths.

This notebook runs **automated scheduling, API and database checks**, not a full manual screen-reader audit or physical-device test. Detailed verification coverage is documented in `docs/REVIEWER-TEST-GUIDE.md` in the public repository.


## Cell 16 — Check resilience and safe failure handling

The next tests verify that the application fails safely when requests cannot be processed:

- Oversized request bodies return **HTTP 413**.
- A temporarily locked SQLite database returns **HTTP 503** instead of partially creating a booking.
- Request identifiers support troubleshooting without disclosing sensitive values such as demo tokens in application logs.

**Expected result:** all targeted error-handling and privacy tests pass, using disposable data.


In [ ]:
# Cell 17 of 23 — Run resilience checks
p=subprocess.run(["node","--test","tests/body-limit-and-contention.test.js","tests/observability-security.test.js"],cwd=ROOT,env=test_env,text=True,capture_output=True)
print(p.stdout[-5000:])
assert p.returncode==0,p.stderr[-2000:]
assert "# fail 0" in p.stdout or "ℹ fail 0" in p.stdout

## Cell 18 — Check database integrity before going live

Before opening the public browser link, this checkpoint verifies the temporary SQLite database and checks that **no two confirmed bookings overlap on the same technician or service bay**.

**Expected result:** `NOTEBOOK_ISOLATED_DB_BASELINE=PASS`, accompanied by the appointment count and overlap result.

This check runs **before** the long-running live cell. Its result stays visible if you stop Cell 23 early. The live cell also attempts to report the final state when it finishes.


In [ ]:
# Cell 19 of 23 — Verify database integrity
# Verify data integrity *before* the interactive session. Colab's Stop button
# interrupts the notebook's Run All operation; later cells would not execute.
import sqlite3

def report_lab_database(label):
    db_path = ROOT / "data" / "scheduler.sqlite"
    assert db_path.is_file(), f"Isolated SQLite database missing: {db_path}"
    with sqlite3.connect(str(db_path), timeout=10) as db:
        integrity = db.execute("PRAGMA integrity_check").fetchone()[0]
        appointments = db.execute("SELECT COUNT(*) FROM appointments").fetchone()[0]
        no_overlap = not db.execute("""
           SELECT 1 FROM appointments a JOIN appointments b
             ON a.id < b.id AND a.status='confirmed' AND b.status='confirmed'
             AND a.start_at_ms < b.end_at_ms AND b.start_at_ms < a.end_at_ms
             AND (a.technician_id=b.technician_id OR a.service_bay_id=b.service_bay_id)
           LIMIT 1
        """).fetchone()
    print(f"{label}=PASS" if integrity == "ok" and no_overlap and appointments > 0 else f"{label}=FAIL",
          "integrity="+integrity, "appointments="+str(appointments),
          "no_overlap="+str(no_overlap), flush=True)
    assert integrity == "ok" and no_overlap and appointments > 0, "SQLite integrity or booking overlap check failed"
    return {"integrity": integrity, "appointments": appointments, "no_overlap": no_overlap}

LAB_BASELINE = report_lab_database("NOTEBOOK_ISOLATED_DB_BASELINE")
print("LIVE_SESSION_BASELINE_READY=PASS — you may safely start the final interactive cell.")


## Cell 20 — Owner access: get the private demo write token

Everyone can browse availability and run the two simulations. **Creating or cancelling appointments requires the owner's private token.** Reviewers should request it privately from the project owner.

**Option A — Terminal while the demo is running:** Open a separate Terminal in Colab (or Kaggle if available). In the web demo, open **Booking access & help → For project owners**, then copy and run the Terminal command. Cell 23 can continue running.

**Option B — Private Python cell before the demo:** Change `REVEAL_TOKEN = False` to `True` in **Cell 21**, run that cell, copy the token privately, then **clear the cell output**. Return the setting to `False` before saving or sharing.

**Already clicked Run All with `REVEAL_TOKEN = False`?** There is no need to rerun the whole notebook. **Stop Cell 23**, change Cell 21 to `True`, run Cell 21, copy the token privately and clear its output. Restore `False`, then run **Cell 23 again** to start a **new temporary Cloudflare URL**.

**Keep credentials private:** Do not include the token in notebooks, saved outputs, GitHub, screenshots, web URLs or public messages. A new isolated setup generates a new token.


In [ ]:
# Cell 21 of 23 — Owner-only token control
# Safe during Run All: this cell never displays the credential by default.
# To reveal after Run All: stop Cell 23, change this to True, run Cell 21, clear output, then rerun Cell 23.
REVEAL_TOKEN = False
assert LAB_TOKEN_FILE.is_file(), "Run setup first"
print("Private demo access is ready. Your token is not displayed.")
print("To retrieve it, see the owner instructions in Cell 20.")
if REVEAL_TOKEN:
    print(LAB_TOKEN_FILE.read_text().strip())
    print("PRIVATE_SECRET_DISPLAYED — copy privately and CLEAR THIS CELL OUTPUT immediately.")


## Cell 22 — Launch the live scheduling demo

**Cell 23 is the final step.** It starts a temporary Cloudflare tunnel and displays **Open Live Scheduling Demo ↗**. Open the link in a new tab.

**During the session:** explore availability and appointments, run conflict/concurrency simulations and—if the owner has authorized access—verify a private write token to confirm or cancel bookings. The notebook streams **sanitized request activity** while the demo is open.

**To end:** stop Cell 23 or wait for the default **15-minute timeout**. The notebook attempts to close the tunnel and local server and report the final database state. Forced runtime termination may prevent cleanup.

**To reopen:** run **Cell 23 again**. The local server can restart using the existing sandbox, and you will receive a **new Cloudflare URL**. Use **Run All** only when you want a brand-new database and fresh setup.

**Need the token?** See **Cells 20–21**. If the final cell is already running and you cannot use Terminal, stop it before executing Cell 21, then restart Cell 23.

Use only synthetic data; never enter real customer information or publish a private token.


In [ ]:
# Cell 23 of 23 — Launch Live Web Demo
# This is the final long-running Cell 23. Stop it before rerunning Cell 21.
# This is intentionally the LAST executable cell. Stopping a cell in Colab
# can abort Run All: there must never be a required report cell after this one.
import html
from IPython.display import HTML, display
import hashlib

LIVE_SESSION_SECONDS = int(os.environ.get("KEYLOOP_LIVE_SESSION_SECONDS", "900"))
assert 1 <= LIVE_SESSION_SECONDS <= 3600, "Choose 1–3600 seconds for the demo"
# The final cell intentionally stops Node on completion. Allow safely rerunning
# this cell against the same isolated SQLite without running all prior tests.
if server is None:
    raise RuntimeError("The isolated server has not been initialized. Run setup and server cells first.")
if server.poll() is not None:
    import socket
    assert Path(sandbox_env["DATABASE_PATH"]).resolve().is_relative_to(ROOT)
    assert Path(sandbox_env["DATABASE_PATH"]).is_file(), "Sandbox SQLite file is missing"
    with socket.socket() as probe:
        if probe.connect_ex(("127.0.0.1", LAB_PORT)) == 0:
            raise RuntimeError("LAB_PORT is occupied. Run All to obtain a fresh sandbox.")
    with open(ROOT / "evidence" / "notebook-server.log", "a") as restart_log:
        server = subprocess.Popen(
            ["node", "src/server.js"], cwd=ROOT, env=sandbox_env,
            stdout=restart_log, stderr=subprocess.STDOUT, start_new_session=True
        )
    for attempt in range(50):
        if server.poll() is not None:
            raise RuntimeError("The isolated server exited during restart")
        try:
            if api("/readyz")[0] == 200:
                break
        except Exception:
            pass
        time.sleep(0.2)
    else:
        raise RuntimeError("Isolated server readiness timeout after restart")
    print("LIVE_SERVER_RESTARTED=PASS", flush=True)
assert server.poll() is None, "Isolated Node server is not running"


live_env = sandbox_env.copy()
live_env["PYTHONUNBUFFERED"] = "1"
script = ROOT / "scripts" / "python" / "Notebook-Live-Tunnel.py"
script.parent.mkdir(parents=True, exist_ok=True)
tunnel_asset = (SOURCE_ROOT / "scripts" / "python" / "Notebook-Live-Tunnel.py").read_bytes()
assert hashlib.sha256(tunnel_asset).hexdigest() == "c03515e6c4fd890df4ac54c683ec10edb8bddf51fc082795c0fa85949f55a8ed", "Tunnel helper asset checksum mismatch"
script.write_bytes(tunnel_asset)
cmd = [sys.executable, "-u", str(script), "--origin-port", str(LAB_PORT),
       "--seconds", str(LIVE_SESSION_SECONDS), "--work-dir", str(ROOT / "evidence")]

live = None
live_failure = None
live_stop_reason = "normal"
try:
    verify_live_source_is_current()
    # Detect wrong web assets immediately before tunnel exposure.
    with urllib.request.urlopen(BASE + "/js/main.js?v=" + SOURCE_REVISION[:16], timeout=15) as resp:
        frontend_response = resp.read().decode("utf-8")
    assert matches_restored_ui(frontend_response), "Live HTTP frontend differs from restored original UI baseline"
    print("LIVE_HTTP_FRONTEND_PREFLIGHT=PASS", flush=True)

    print("LIVE_INTERACTIVE_SESSION=STARTING", flush=True)
    print("Open the live link below; stop Cell 23 when you finish.", flush=True)
    live = subprocess.Popen(cmd, cwd=ROOT, env=live_env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1,
                            start_new_session=True)
    for line in live.stdout:
        print(line, end="", flush=True)
        if line.startswith("LIVE_WEB_UI_URL="):
            live_url = line.split("=", 1)[1].strip()
            display(HTML('<p><a target="_blank" rel="noopener noreferrer" href="{}"><strong>Open Live Scheduling Demo ↗</strong></a></p>'.format(html.escape(live_url, quote=True))))
    rc = live.wait(timeout=10)
    if rc != 0:
        raise RuntimeError(f"Live tunnel terminated with code {rc}; inspect log output above")
    print("LIVE_SESSION_FINISHED=PASS", flush=True)
except KeyboardInterrupt:
    # Colab's Stop button sends an interrupt. Convert it into normal completion,
    # so post-interaction SQLite reporting and process cleanup still run.
    live_stop_reason = "user_stop"
    print("\nLIVE_SESSION_STOPPED_BY_USER=PASS", flush=True)
    print("Reporting SQLite and closing tunnel, proxy and Node server...", flush=True)
except Exception as exc:
    live_failure = exc
    live_stop_reason = "error"
    print("LIVE_SESSION_ERROR=", type(exc).__name__, str(exc)[:350], flush=True)
finally:
    # The helper owns cloudflared and the proxy and handles SIGTERM in its finally.
    # Kill the process group only when necessary; never leave a public tunnel up.
    if live is not None:
        try:
            if live.poll() is None:
                try:
                    os.killpg(live.pid, signal.SIGTERM)
                    live.wait(timeout=7)
                except (ProcessLookupError, subprocess.TimeoutExpired):
                    if live.poll() is None:
                        os.killpg(live.pid, signal.SIGKILL)
                        live.wait(timeout=5)
        except Exception as cleanup_exc:
            print("LIVE_TUNNEL_CLEANUP_WARNING=", type(cleanup_exc).__name__, flush=True)
        finally:
            if live.stdout is not None:
                try:
                    live.stdout.close()
                except Exception as stream_exc:
                    print("LIVE_LOG_STREAM_CLOSE_WARNING=", type(stream_exc).__name__, flush=True)

    # Report in THIS cell, not a subsequent one. The baseline above survives
    # even if the Colab UI forcibly terminates the active execution.
    try:
        LAB_FINAL = report_lab_database("NOTEBOOK_ISOLATED_DB")
    except Exception as report_exc:
        if live_failure is None:
            live_failure = report_exc
        print("SQLITE_FINAL_REPORT_FAILED=", type(report_exc).__name__, flush=True)
    finally:
        try:
            stop_lab_server()
            assert server is not None and server.poll() is not None
            print("ISOLATED_SERVER_STOPPED=PASS", flush=True)
        except Exception as server_exc:
            print("ISOLATED_SERVER_STOPPED=FAIL", type(server_exc).__name__, flush=True)
            if live_failure is None:
                live_failure = server_exc
    print("LIVE_SESSION_CLEANUP_COMPLETE=PASS" if live_failure is None else
          "LIVE_SESSION_CLEANUP_COMPLETE_WITH_ERRORS", "reason="+live_stop_reason,
          flush=True)

if live_failure is not None:
    raise live_failure
